# Mecánica y paisajes de energía

Explora potenciales periódicos, coordenadas conformacionales, barreras energéticas, superficies 2D/3D y el efecto de la carga externa.

> **Nota científica:** ProteinMotorSim es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

In [ ]:
from pathlib import Path
import sys

# Busca la raíz del proyecto para poder importar src/protein_motor_sim.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "protein_motor_sim").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No encuentro la carpeta src/protein_motor_sim. "
        "Abre JupyterLab desde la raíz de ProteinMotorSim o instala el paquete "
        "con python -m pip install -e ."
    )

print(f"Proyecto detectado en: {PROJECT_ROOT}")

## 1. Qué representa el paisaje

Un paisaje `U(x, q)` representa una energía potencial ilustrativa en función de una coordenada axial `x` (nm) y una coordenada conformacional adimensional `q` entre 0 y 1. Los resultados se expresan en unidades de `k_B T`. Los parámetros son fenomenológicos: no se deducen de las tasas de Gillespie ni son una superficie de energía libre experimental de kinesina.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from protein_motor_sim.mechanics.energy_landscape import EnergyLandscape
from protein_motor_sim.visualization.energy_maps import (
    plot_energy_landscape_2d, plot_energy_landscape_3d, plot_energy_profiles
)
from protein_motor_sim.visualization.energy_landscape import (
    plot_periodic_energy_landscape, plot_conformational_energy_surface,
    plot_state_energy_diagram, thermal_energy_pN_nm
)

## 2. Construir un paisaje bidimensional

La superficie combina: periodicidad axial, doble pozo conformacional, sesgo, acoplamiento y una inclinación por carga externa. La carga opuesta añade un término lineal proporcional a `F x / (k_B T)`.

In [ ]:
landscape = EnergyLandscape(
    period_nm=8.2,
    spatial_barrier_kBT=2.5,
    conformational_barrier_kBT=3.0,
    conformational_bias_kBT=0.0,
    phase_shift=0.5,
    coupling_kBT=0.8,
    opposing_force_pN=0.0,
    temperature_K=310.0,
)
print("kBT a 310 K:", thermal_energy_pN_nm(310.0), "pN nm")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_energy_landscape_2d(landscape, x_range_nm=(-8.2, 32.8), ax=ax)
plt.show()

## 3. Superficie tridimensional

La tercera dimensión es la energía. Esta vista ayuda a reconocer valles, barreras y regiones conformacionales favorecidas.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
plot_energy_landscape_3d(landscape, x_range_nm=(-8.2, 32.8), ax=ax)
plt.show()

## 4. Perfiles a conformación fija

Al fijar `q`, observamos cómo cambia el potencial a lo largo de `x`. Se puede comparar el estado conformacional inicial, intermedio y final.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_energy_profiles(
    landscape,
    x_range_nm=(-8.2, 32.8),
    conformations={"q inicial": 0.0, "intermedio": 0.5, "q final": 1.0},
    ax=ax,
)
plt.show()

## 5. Cómo inclina el paisaje la carga externa

Compara potenciales periódicos sometidos a distintas fuerzas opuestas. Una carga positiva hace crecer la energía hacia `+x` en la convención de este módulo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_periodic_energy_landscape(
    x_range_nm=(0, 32.8),
    lattice_spacing_nm=8.2,
    barrier_kbt=4.0,
    forces_pN=(0.0, 1.0, 2.0, 4.0),
    temperature_K=310.0,
    ax=ax,
)
plt.show()

## 6. Diagrama esquemático de estados y barreras

Este diagrama requiere introducir energías relativas y barreras de forma explícita. No es correcto inferirlas directamente de las tasas actuales, porque todavía no imponemos balance detallado local.

In [ ]:
state_energies = {
    "bound_apo": 0.0,
    "atp_bound": -1.0,
    "post_hydrolysis": -0.2,
}
barriers = {
    ("bound_apo", "atp_bound"): 2.5,
    ("atp_bound", "post_hydrolysis"): 2.2,
}
fig, ax = plt.subplots(figsize=(8, 4))
plot_state_energy_diagram(state_energies, barriers_kbt=barriers, ax=ax)
plt.show()

## 7. Interpretar el resultado

- Un mínimo corresponde a una configuración relativamente estable en el potencial elegido.
- Una barrera alta dificulta una transición térmica.
- La carga inclina el paisaje y cambia el trabajo mecánico asociado al desplazamiento.
- El mapa es un modelo matemático de exploración, no un potencial de fuerza molecular ni una energía libre medida de kinesina.